# Phase 2: Content-Based Course Recommendation

In this notebook, we build and evaluate our first personalized recommendation model: **Content-Based Filtering**.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath('../src'))
from content_based import ContentBasedRecommender, build_combined_course_features
from preprocessing import PopularityRecommender

# Load datasets
df_users = pd.read_csv('../data/raw/users.csv')
df_courses = pd.read_csv('../data/raw/courses.csv')
df_interactions = pd.read_csv('../data/raw/interactions.csv')

print(f"Loaded {len(df_users)} users, {len(df_courses)} courses, and {len(df_interactions)} interactions.")

## 1. Item-to-Item Similarity Recommendation
Let's test `ContentBasedRecommender.recommend_similar_courses()` for Course `C06` (Python Programming Masterclass).

In [ ]:
recommender = ContentBasedRecommender()
recommender.fit(df_courses, df_interactions)

similar_c06 = recommender.recommend_similar_courses('C06', top_n=5)
similar_c06[['course_id', 'title', 'category', 'similarity_score']]

## 2. Personalized User Recommendations
Let's test recommendations for User `U001` (whose persona is Frontend Developer).

In [ ]:
user_recs_u001 = recommender.recommend('U001', top_n=5)
user_recs_u001[['course_id', 'title', 'category', 'similarity_score', 'recommendation_type']]

## 3. Cold-Start Fallback Demonstration
For a brand new user `U999` with zero interaction history, our recommender falls back to the Popularity Baseline.

In [ ]:
cold_start_recs = recommender.recommend('U999', top_n=5)
cold_start_recs[['course_id', 'title', 'category', 'recommendation_type']]

## 4. Offline Evaluation: Leave-Last-Course-Out Validation (Precision@5 & Recall@5)

In [ ]:
# Perform Leave-Last-Course-Out Split per user
train_rows = []
test_lookup = {}

for user_id, group in df_interactions.groupby('user_id'):
    unique_courses = group['course_id'].unique()
    if len(unique_courses) >= 2:
        # Last unique course becomes test set
        held_out_course = unique_courses[-1]
        test_lookup[user_id] = held_out_course
        # All previous course interactions form training set
        train_rows.append(group[group['course_id'] != held_out_course])
    else:
        train_rows.append(group)

df_train_interactions = pd.concat(train_rows, ignore_index=True)

# Fit Recommenders on Train Set
cb_model = ContentBasedRecommender().fit(df_courses, df_train_interactions)
pop_model = PopularityRecommender()
pop_model.fit(cb_model.user_interaction_weights, df_courses)

# Evaluate Precision@5 and Recall@5
cb_hits, pop_hits = 0, 0
eval_users = len(test_lookup)

for uid, target_course in test_lookup.items():
    cb_recs = cb_model.recommend(uid, top_n=5)['course_id'].tolist()
    pop_recs = pop_model.recommend(uid, top_n=5)['course_id'].tolist()
    
    if target_course in cb_recs:
        cb_hits += 1
    if target_course in pop_recs:
        pop_hits += 1

cb_precision = (cb_hits / (eval_users * 5))
cb_recall = (cb_hits / eval_users)
pop_precision = (pop_hits / (eval_users * 5))
pop_recall = (pop_hits / eval_users)

print(f"=== EVALUATION RESULTS (N={eval_users} Users) ===")
print(f"Content-Based Recommender -> Precision@5: {cb_precision:.4f} | Recall@5: {cb_recall:.4f}")
print(f"Popularity Baseline       -> Precision@5: {pop_precision:.4f} | Recall@5: {pop_recall:.4f}")